# Stitch Integration

This notebook demonstrates the **Stitch** stage of the Foreign Whispers dubbing pipeline.
It performs final video assembly: combining the original video with dubbed TTS audio and
rolling two-line translated captions via ffmpeg. The stitch uses audio-only remux
(no re-encoding), preserving original video quality.

**Prerequisites:**
- The Docker stack must be running (`docker compose --profile nvidia up -d`).
- The API should be accessible at `http://localhost:8082`.
- Prior pipeline stages (download, transcribe, translate, TTS) must have completed for the target video.

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

# Load .env (LOGFIRE_TOKEN, HF_TOKEN, etc.)
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from foreign_whispers.client import FWClient, BASELINE

fw = FWClient("http://localhost:8080")
fw.healthz()

# Optional: Logfire tracing (no-op shim if unavailable)
try:
    import logfire
    logfire.configure(service_name="foreign-whispers-stitch")
    print("Logfire tracing enabled.")
except Exception:
    class _NoopSpan:
        def __enter__(self): return self
        def __exit__(self, *a): pass
    class _noop:
        @staticmethod
        def span(name, **kw): return _NoopSpan()
        @staticmethod
        def info(*a, **kw): pass
    logfire = _noop()
    print("Logfire not configured — using no-op shim.")

## Stitch Video

Call the API to combine the original video with the dubbed TTS audio track.
The stitch endpoint replaces the audio stream via ffmpeg remux (no video re-encoding)
and generates rolling two-line VTT captions.

In [ ]:
video_id = "GYQ5yGV_-Oc"  # replace with your video ID

with logfire.span("stitch", video_id=video_id):
    result = fw.stitch(video_id)

print(f"Video ID:   {result['video_id']}")
print(f"Video path: {result['video_path']}")
print(f"Config:     {result['config']}")

## Inspect Output Artifacts

The stitch stage produces files in two directories:

- `pipeline_data/api/dubbed_videos/{config}/` — final dubbed MP4 files
- `pipeline_data/api/dubbed_captions/` — target-language VTT caption files

In [ ]:
dubbed_videos_dir = PROJECT_ROOT / "pipeline_data" / "api" / "dubbed_videos"
dubbed_captions_dir = PROJECT_ROOT / "pipeline_data" / "api" / "dubbed_captions"

print("Dubbed videos:")
for f in sorted(dubbed_videos_dir.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.relative_to(dubbed_videos_dir)}  ({size_mb:.1f} MB)")

print()
print("Dubbed captions:")
for f in sorted(dubbed_captions_dir.rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.relative_to(dubbed_captions_dir)}  ({size_kb:.1f} KB)")

## View Generated Captions

The stitch stage generates VTT captions in a rolling two-line format:
the current translated line appears on top, and the previous line is shown
on the bottom, giving viewers context continuity.

In [ ]:
# Find the VTT file for this video
vtt_files = list(dubbed_captions_dir.rglob(f"{video_id}*.vtt"))
if not vtt_files:
    vtt_files = list(dubbed_captions_dir.rglob("*.vtt"))

if vtt_files:
    vtt_path = vtt_files[0]
    print(f"Caption file: {vtt_path.name}\n")
    lines = vtt_path.read_text().splitlines()
    for line in lines[:30]:
        print(line)
    if len(lines) > 30:
        print(f"\n... ({len(lines) - 30} more lines)")
    print()
    print("Note the rolling two-line pattern: each cue shows the current")
    print("translated line on top and the previous line on the bottom,")
    print("providing continuity for the viewer.")
else:
    print("No VTT files found. Run the stitch step first.")

## Playback

To play the dubbed output:

1. **Frontend:** Open <http://localhost:8501> and select the video from the list.
   The UI will load the dubbed MP4 with captions overlay.

2. **Direct file:** Play the MP4 directly from
   `pipeline_data/api/dubbed_videos/{config}/{video_id}.mp4` using any media player
   (e.g., VLC, mpv). Load the corresponding VTT file from `pipeline_data/api/dubbed_captions/`
   as an external subtitle track.

## Summary

The stitch stage produced:

- **Dubbed MP4** in `pipeline_data/api/dubbed_videos/{config}/` — original video with replaced audio track
- **VTT captions** in `pipeline_data/api/dubbed_captions/` — rolling two-line translated subtitles

Audio-only remux means no quality loss on the video track: ffmpeg copies the video
stream as-is and only replaces the audio stream with the synthesized TTS output.